# Лабораторная работа 6

# Выравнивание списка, состоящего из итерируемых объектов, на основе рекурсии

Немчинович Виктория Дмитриевна, 27.04.2025

## Постановка задачи

$\color{red}\text{Напишите}$ пользовательскую функцию `flatten_it(nested_obj, gen)`
выравнивания списка на основе рекурсии, которая для списка `nested_obj`, содержащего различные итерируемые объекты (строка, список, кортеж, множество,
словарь), возвращает либо список из всех элементов итерируемых объектов без сохранения структуры итерируемых объектов, если `gen=False`, либо функция возвращает генераторный объект, если `gen=True`, который генерирует по запросу элементы итерируемых объектов при итерационном проходе по исходному списку.

In [4]:
import sys
sys.getrecursionlimit()

3000

In [5]:
def sum(n):
 """суммирует числа от 0 до n>=0"""
 match n:
     case 0: return n # база рекурсии
     case _: return n + sum(n-1)
sum(0), sum(2), sum(10)

(0, 3, 55)

In [6]:
sum = lambda n: n + sum(n-1) if n else n
sum(0), sum(2), sum(10)

(0, 3, 55)

In [7]:
def flatten_it(nested_obj, gen=False):
    def _flatten(obj):
        if isinstance(obj, str) and len(obj) == 1:
            yield obj
        elif isinstance(obj, str):
            yield obj
        elif isinstance(obj, dict):
            for key, value in obj.items():
                yield from _flatten(key)
                yield from _flatten(value)
        elif isinstance(obj, (list, tuple, set)):
            for item in obj:
                yield from _flatten(item)
        else:
            yield obj

    flattened_generator = _flatten(nested_obj)
    return flattened_generator if gen else list(flattened_generator)

Приведем различные примеры с результатами вызовов функции `flatten_it`.

Пример 1. Если вложенным итерируемым объектом является строка, то функция `flatten_it` оставляет строку без изменений, не итерируя ее по символам.

In [8]:
flatten_it(["abc",["de"]], gen=False)

['abc', 'de']

Пример 2. Если вложенным итерируемым объектом является список или кортеж или множество, то функция `flatten_it` возвращает список, состоящий из элементов списка,кортежа и множества без сохранения структуры вложенности для элементов.

In [9]:
flatten_it([1,(2,[3]),[{4}]], gen=False)

[1, 2, 3, 4]

Пример 3. Если вложенным итерируемым объектом является словарь, то функция `flatten_it` сохраняет ключи и соответствующие им объекты в качестве элементов результирующего списка.

In [10]:
flatten_it([{1: {2,3}, 4: [5]}], gen=False)

[1, 2, 3, 4, 5]

Пример 4. Для полученного вложенного списка параметр `gen=True` указывает, что нужно вернуть генератор, а не список. При первом вызове `next()` генератор возвращает значение `1`, приостанавливается, запоминая текущую позицию в списке и все локальные переменные. Таким образом память содержит только текущий элемент `1`, и вложенный список `[2,[3]]` не обработан. Если следующие `next()` не вызывать, то элементы `2` и `3` никогда не будут вычислены и генератор сохранит "замороженное" состояние.

In [11]:
gen = flatten_it([1,[2,[3]]], gen=True)
print(next(gen))

1


## Задание 6.1. Выравнивание вложенных списков

### Задание 6.1.1. Рекурсивная функция flatten_list_v1

Предположим, что нам нужно выровнять вложенный список

In [12]:
it_list = [[1],[2,[3]],[[[[[4]]]]],5,6,7]

Напишем рекурсивную функцию `flatten_list_v1(nested_obj)`, которая выравнивает вложенный список `nested_obj` и возвращает список, содержащий только элементы исходного списка без сохранения его структуры. 

Некоторые аспекты реализации:
- для проверки, является ли текущий объект `item` списком или нет, будем использовать функцию `isinstance(item, list)`;
- для формирования результирующего списка `result` будем использовать дополненное присваивание `+=` для списков на основе операции конкатенации `+`.

In [13]:
?isinstance

Signature: isinstance(obj, class_or_tuple, /)
Docstring:
Return whether an object is an instance of a class or of a subclass thereof.

A tuple, as in ``isinstance(x, (A, B, ...))``, may be given as the target to
check against. This is equivalent to ``isinstance(x, A) or isinstance(x, B)
or ...`` etc.
Type:      builtin_function_or_method

In [14]:
def flatten_list_v1(nested_obj):
    result = []  # пустой список для результата
    if isinstance(nested_obj, list):  # проверка, является ли аргумент списком
        for item in nested_obj:  # если аргумент -- список, то перебираем все его элементы
            result += flatten_list_v1(item) # рекурсия
    else:  # если аргумент не список
        result += [nested_obj]  # добавляем элемент в список
    return result  # возвращаем итоговый результат

In [15]:
flatten_list_v1(it_list)

[1, 2, 3, 4, 5, 6, 7]

### Задание 6.1.2. Генераторная функция на основе рекурсии flatten_list_v2

На основе функции `flatten_list_v1` напишите генераторную функцию `flatten_list_v2(nested_obj)`, которая выравнивает вложенный список и возвращает генератнорый объект для элементов вложенного списка.

Реализация генераторной функции `flatten_list_v2` будет проще реализации функции `flatten_list_v1`, так как не нужно объединять все объекты в единый результирующий список.

$\color{red}\text{Предложите два варианта реализации}$: только с использованием `yield`(вариант 1) и с использованием `yield from`(вариант 2).

Вариант 1 (только с использованием `yield`)

In [16]:
def flatten_list_v2(nested_obj):
    if isinstance(nested_obj, list):  # проверка, является ли аргумент списком
        for item in nested_obj:  # если аргумент -- список, то перебираем все его элементы
            for sub_item in flatten_list_v2(item):  # рекурсивный вызов
                yield sub_item  # возврат подэлемента
    else:  # если аргумент не список
        yield nested_obj  # возврат простого элемента

In [17]:
[x for x in flatten_list_v2(it_list)] 

[1, 2, 3, 4, 5, 6, 7]

Вариант 2 (с использованием `yield from`)

In [18]:
def flatten_list_v2(nested_obj):
    if isinstance(nested_obj, list):  # проверка, является ли аргумент списком
        for item in nested_obj:  # если аргумент -- список, то перебираем все его элементы
            yield from flatten_list_v2(item)  # рекурсивный вызов с yield from
    else:  # если аргумент не список
        yield nested_obj  # возврат простого элемента

In [19]:
[x for x in flatten_list_v2(it_list)] 

[1, 2, 3, 4, 5, 6, 7]

### Задание 6.1.3. Функция flatten_list: создание, документирование, тестирование

In [20]:
def flatten_list(nested_obj, gen = False):
    """формирует список или генераторный объект
    из вложенных списков без сохранения структуры
        Arguments :
    nested_obj : список, которой может содержать в себе все итерируемые объекты, структура сохранится всех итерируемых объектов, кроме типа list
    gen : хранит булевскую переменную, в зависимости от которой будет выведен соответствующий результат
        Returns : генераторный объект, если gen = True, список из элементов вложенных списков без сохранения структуры, если gen = False. Default gen = False.
    """
    if gen == False:                             # проверка на значение аргумента gen
        return flatten_list_v1(nested_obj)         # вернуть выполнение функции flatten_list1, если условие = False 
    else:                                        # иначе
        return flatten_list_v2(nested_obj)         # вернуть выполнение функции flatten_list2, если условие = True

In [21]:
flatten_list.__doc__

'формирует список или генераторный объект\n    из вложенных списков без сохранения структуры\n        Arguments :\n    nested_obj : список, которой может содержать в себе все итерируемые объекты, структура сохранится всех итерируемых объектов, кроме типа list\n    gen : хранит булевскую переменную, в зависимости от которой будет выведен соответствующий результат\n        Returns : генераторный объект, если gen = True, список из элементов вложенных списков без сохранения структуры, если gen = False. Default gen = False.\n    '

In [22]:
flatten_list?

Signature: flatten_list(nested_obj, gen=False)
Docstring:
формирует список или генераторный объект
из вложенных списков без сохранения структуры
    Arguments :
nested_obj : список, которой может содержать в себе все итерируемые объекты, структура сохранится всех итерируемых объектов, кроме типа list
gen : хранит булевскую переменную, в зависимости от которой будет выведен соответствующий результат
    Returns : генераторный объект, если gen = True, список из элементов вложенных списков без сохранения структуры, если gen = False. Default gen = False.
File:      c:\users\user\appdata\local\temp\ipykernel_2788\1546477758.py
Type:      function

In [23]:
help(flatten_list)

Help on function flatten_list in module __main__:

flatten_list(nested_obj, gen=False)
    формирует список или генераторный объект
    из вложенных списков без сохранения структуры
        Arguments :
    nested_obj : список, которой может содержать в себе все итерируемые объекты, структура сохранится всех итерируемых объектов, кроме типа list
    gen : хранит булевскую переменную, в зависимости от которой будет выведен соответствующий результат
        Returns : генераторный объект, если gen = True, список из элементов вложенных списков без сохранения структуры, если gen = False. Default gen = False.



#### $\color{red}{Тестирование}$ функции `flatten_list`:

In [24]:
my_it_list = [1,[[[[2]]],3],[(4,6)]]

print(flatten_list(my_it_list))        # аргумент по умолчанию gen = False
print(flatten_list(my_it_list, False)) # указываем явно аргумент gen = False

print(flatten_list(my_it_list, True))  # указываем явно аргумент gen = True. Возвращает генераторный объект
[x for x in flatten_list(my_it_list, True)]  # с помощью итерационного инструмента list(списковое включение) считываем генераторный объект и выводим его значение

[1, 2, 3, (4, 6)]
[1, 2, 3, (4, 6)]
<generator object flatten_list_v2 at 0x000001BCCD08C120>


[1, 2, 3, (4, 6)]

In [25]:
my_it_list = [1,[[[[2]]],3],[(4,6)]]
iter_obj = iter(flatten_list(my_it_list))
rez = flatten_list(my_it_list, True)
#print([iter_obj.__next__() for _ in flatten_list(my_it_list)])
[x for x in flatten_list(my_it_list)]
iter_obj.__next__(), iter_obj.__next__(), iter_obj.__next__(), iter_obj.__next__()

(1, 2, 3, (4, 6))

## Задание 6.3. Обработка циклических объектов

Циклическими объектами в Python могут быть списки и словари. Циклический
список содержит элемент, который ссылается на весь список. Циклический словарь
содержит элемент <ключ>:<значение>, <значение> которого сылается на весь
словарь.

Создадим циклический объект списка

In [26]:
it_list_cyclic = it_list[:]
it_list_cyclic.append(it_list_cyclic)

Для отображения циклических объектов в Python используется специальный
синтаксис

In [27]:
it_list_cyclic

[[1], [2, [3]], [[[[[4]]]]], 5, 6, 7, [...]]

Обработаем ситуацию выравнивания вложенного списка из Задания 6.1, когда
вложенный список является циклическим объектом

Для этого изменим код пользовательской функции flatten_list_v1 из Задания
6.1, добавив генерацию исключения ValueError с помощью оператора raise ,
когда текущий элемент item списка nested_obj ссылается на этот же список

In [28]:
def flatten_list_v1(nested_obj):
    result = []
    if isinstance(nested_obj,list):
        for item in nested_obj:
            if item is nested_obj:
                raise ValueError("Cyclic list is found") # генерация исключения
            else:
                result += flatten_list_v1(item)
    else:
        result += [nested_obj]
    return result

Протестируем функцию flatten_list_v1

In [29]:
flatten_list_v1(it_list)

[1, 2, 3, 4, 5, 6, 7]

In [30]:
flatten_list_v1(it_list_cyclic)

ValueError: Cyclic list is found

Для того, чтобы стандартное сообщение об ошибке не выводилось, перехватим
созданное нами исключение ValueError внутри функции flatten_list_v1 с
использованием оператора try в сочетании с except . Если внутри блока try
возникнет исключение ValueError , то процесс выполнения кода перейдет в блок
оператора except , для которого указано исключение ValueError

In [ ]:
try:
 print(flatten_list_v1(it_list_cyclic))
except ValueError as e:
 print(e)

 ### Задание 6.3.1. Доопределение функции flatten_it


$\color{red}{Доопределите}$ пользовательскую функцию flatten_it из Задания 6.2, добавив
генерацию исключения ValueError с помощью оператора raise при обработке
циклических списков и циклических словарей.

In [ ]:
def flatten_it(nested_obj, gen=False):
    def _flatten(obj, visited=None):
        if visited is None:
            visited = set()
        
        if isinstance(obj, (list, dict, set)):
            obj_id = id(obj)
            if obj_id in visited:
                raise ValueError('Cyclic reference detected')
            visited.add(obj_id)
        
        if isinstance(obj, str):
            yield obj
        elif isinstance(obj, dict):
            for key, value in obj.items():
                yield from _flatten(key, visited)
                yield from _flatten(value, visited)
        elif isinstance(obj, (list, tuple, set)):
            for item in obj:
                yield from _flatten(item, visited)
        else:
            yield obj

    try:
        flattened_generator = _flatten(nested_obj)
        return flattened_generator if gen else list(flattened_generator)
    except ValueError as e:
        raise ValueError(f'Cyclic object detected in input structure: {e}')

### Задание 6.3.2. Тестирование функции flatten_it

1. Циклический список(ссылается сам на себя)

In [ ]:
cyclic_list = [1, 2, 3]
cyclic_list.append(cyclic_list)

try:
    list(flatten_it(cyclic_list))
except ValueError as e:
    print(e)

2. Вложенные списки и циклы

In [ ]:
a = [1, 2]
b = [3, 4, a]
a.append(b)

try:
    list(flatten_it(a))
except ValueError as e:
    print(e)

3. Словарь с циклической ссылкой

In [ ]:
cyclic_dict = {"a": 1, "b": 2}
cyclic_dict["self"] = cyclic_dict

try:
    list(flatten_it(cyclic_dict))
except ValueError as e:
    print(e)
else:
    print("Словарь не вызвал ошибку (функция не проверяет словари)")

$\color{red}{Задание*(необязательное)}$ Поясните, почему код не работает для следующего циклического объекта

In [ ]:
test_cyclic = {'a' : 1}
test_cyclic['b'] = {2 : test_cyclic}
print(test_cyclic)

Функция flatten_it не обнаруживает циклические ссылки во вложенных словарях, потому что она проверяет только id самого внешнего словаря, но не его внутренних элементов. Когда вложенный словарь {2: test_cyclic} создаётся, он получает новый id, и связь с исходным test_cyclic остаётся незамеченной 